# Colab Baseline Runner (Costa)

This notebook uses Colab as execution backend and runs ML baselines with DagsHub/MLflow + Optuna SQLite persistence.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Set these for your session (or use Colab secrets)
import os
os.environ['DAGSHUB_USERNAME'] = 'aminetech26'
os.environ['DAGSHUB_REPO'] = 'PFE_Experiments'
os.environ['DAGSHUB_USER_TOKEN'] = '5e31845f92874871e830dd2f59859e3d632c1aa0'

In [ ]:
%cd /content
!git clone https://github.com/aminetech26/PFE_Experiments.git
%cd /content/PFE_Experiments/PFE_Experiments

## DVC-first data materialization (recommended with DagsHub)

In [ ]:
# Pull tracked data artifacts from DagsHub storage
!uv run dvc pull

In [ ]:
# Reproduce data pipeline stages if needed (Costa)
!uv run dvc repro ingest
!uv run dvc repro split
!uv run dvc repro preprocess
!uv run dvc repro featurize

In [ ]:
!pip install -q uv
!uv sync

In [ ]:
# Initialize Drive folders and show config snippet
!uv run python -m src.training.colab_dl_template --init --status --show-config

In [ ]:
# Optional health check (MLflow + Optuna)
!uv run python scripts/check_tracking_health.py

In [ ]:
# Set one seed in one place (global)
import yaml
from pathlib import Path
cfg_path = Path('configs/model_config.yaml')
cfg = yaml.safe_load(cfg_path.read_text(encoding='utf-8'))
cfg['experiment']['seed'] = 42  # change per repeated run
cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding='utf-8')
print('experiment.seed set to', cfg['experiment']['seed'])

## Baseline Commands (edit model/profile/split as needed)

In [ ]:
# Classification baseline (implemented now: lightgbm)
!uv run python -m src.modeling.classification.ml.run --model lightgbm --dataset costa --split-path path_a --profile baseline_raw --run-type baseline

In [ ]:
# Anomaly baseline (implemented now: matrix_profile)
!uv run python -m src.modeling.anomaly_detection.ml.run --model matrix_profile --dataset costa --split-path path_a --profile baseline_raw --run-type baseline

In [ ]:
# Handcrafted comparator example (plus_physics)
!uv run python -m src.modeling.classification.ml.run --model lightgbm --dataset costa --split-path path_a --profile plus_physics --run-type baseline
!uv run python -m src.modeling.anomaly_detection.ml.run --model matrix_profile --dataset costa --split-path path_a --profile plus_physics --run-type baseline